# 关于本 Notebook

## 推理后端（Inference Backends）：冷启动、缓存与部署物理约束

当你的智能体调用一个模型时，它并不只是和“某个 API”对话。真正响应请求的是一个 **GPU 进程**，而这个进程受到真实的系统物理约束：模型必须先加载进 VRAM，才能生成第一个 token；每个新的 Prompt 前缀都需要计算 KV cache（KV 缓存）；显存还需要不断分配、管理，并在必要时驱逐已有缓存。

这些部署现实，决定了你的智能体究竟能在 200ms 内开始响应，还是要等上 12 秒：

| 关注点 | 实际发生什么 | 对智能体的影响 |
|---|---|---|
| **冷启动（Cold Start）** | 模型权重从磁盘 / 网络加载到 GPU VRAM | 部署后的首次请求可能需要 30–120 秒，而不是 <1 秒 |
| **KV Cache（KV 缓存）** | Prompt 中每个 token 都要计算 attention 的 key/value | 如果没有缓存，长 system prompt 会在每次调用时重新计算 |
| **前缀缓存（Prefix Caching）** | 多个请求复用共享 Prompt 前缀对应的 KV cache | 相同 system prompt 可跳过重复计算，TTFT（Time to First Token，首 token 延迟）可改善约 2–5 倍 |
| **缓存失效（Cache Invalidation）** | 前缀变化后，原有 KV 缓存无法复用 | 新 system prompt 或对话分叉会带来冷缓存延迟尖峰 |
| **显存压力（Memory Pressure）** | KV cache 会随上下文长度 × batch size 增长 | 长对话可能挤掉其他请求的缓存，推高尾延迟 |

本 Notebook 会启动一个本地 **vLLM** 推理服务器，然后使用真实请求逐项测量这些影响——不只讲概念，而是直接观察数值。

我们还会比较三种主流开源推理后端：**vLLM**、**TGI** 和 **SGLang**，帮助你判断不同场景下该选择哪一个。

**冷启动是一种部署税（deployment tax），不是 Bug。**
每次重新部署、从零扩容，或者崩溃后重启，都会出现约 30–120 秒的不可服务窗口。常见缓解方式包括 keep-alive 探针、预热请求（pre-warming requests），以及把最小副本数设为大于 0。

**KV cache 就像智能体的工作记忆，而且代价很高。**
对话越长，消耗的 VRAM 越多，TTFT 越慢，可同时服务的用户也越少。因此，前一个 Notebook 里的上下文裁剪（context pruning）不只是成本优化，同样也是延迟优化。

**前缀缓存（Prefix Caching）几乎没有额外成本，却能显著降低 TTFT。**
如果智能体使用稳定的 system prompt——通常也应该如此——就应启用 prefix caching。但动态前缀内容，例如 RAG 片段和用户画像，会让缓存失效。因此 Prompt 模板最好采用 **稳定前缀（stable prefix）+ 可变后缀（variable suffix）** 的结构。

**缓存失效（Cache Invalidation）是灵活性的隐性成本。**
每次修改 system prompt、让对话产生分叉，或者在 Prompt 顶部注入新的上下文，都需要重新支付完整的 KV 计算成本。这就是智能体灵活性与推理效率之间的权衡。

**智能体代码应与具体推理后端解耦（backend-agnostic）。**
可以把 OpenAI SDK 作为统一客户端。vLLM、TGI、SGLang 以及许多云端服务商都提供 OpenAI-compatible API（OpenAI 兼容接口），因此切换后端通常只需要改配置，而不是重构智能体代码。

```
生产部署检查清单：
  ☐  已选择推理后端（vLLM / TGI / SGLang）
  ☐  已启用前缀缓存
  ☐  System prompt 已按稳定前缀设计
  ☐  已测量并记录冷启动时间
  ☐  已监控健康检查端点
  ☐  最小副本数 > 0（避免 scale-to-zero 导致冷启动）
  ☐  已配置上下文裁剪（最大历史 token 数）
  ☐  已估算 GPU 显存预算（模型 + KV cache + 额外开销）
```

# 推理后端（Inference Backends）：vLLM vs. TGI vs. SGLang

三者都是开源、面向 GPU 的推理服务器，并且都能暴露 OpenAI-compatible API。你的智能体代码不需要变化——`client.chat.completions.create()` 都可以使用；真正变化的是它们的**运行特征（operational profile）**。

| 特性 | **vLLM** | **TGI**（Text Generation Inference） | **SGLang** |
|---|---|---|---|
| **主要维护方** | UC Berkeley（开源社区） | Hugging Face | LMSYS |
| **调度机制** | PagedAttention（高显存效率 KV） | Continuous batching（连续批处理） | RadixAttention（基于树的 KV 复用） |
| **前缀缓存** | `--enable-prefix-caching`（APC） | ✗（只能手动做 Prompt 缓存） | 内置（RadixAttention） |
| **多 GPU** | Tensor parallel + pipeline parallel | Tensor parallel | Tensor parallel + expert parallel |
| **推测解码（Speculative Decoding）** | ✓（draft model 或 n-gram） | ✓（Medusa heads） | ✓ |
| **量化（Quantization）** | AWQ、GPTQ、FP8、BitsAndBytes | AWQ、GPTQ、EETQ、BitsAndBytes | AWQ、GPTQ、FP8 |
| **结构化输出（Structured Output）** | ✓（guided decoding / outlines） | ✓（grammar-based） | ✓（compressed FSM） |
| **视觉模型** | ✓ | ✓ | ✓ |
| **更适合** | 通用、高吞吐场景 | HF 生态、快速部署 | 多轮智能体（树状缓存） |
| **冷启动** | ~30–120s（取决于模型） | ~20–90s | ~30–120s |
| **API 兼容性** | OpenAI-compatible | OpenAI-compatible + TGI native | OpenAI-compatible + SGLang native |

### 什么时候用哪一个

- **vLLM**：默认选择。文档完善、模型覆盖最广，PagedAttention 也经过了大量生产验证。除非有明确原因，否则优先从它开始。

- **TGI**：当你已经深度使用 Hugging Face 生态，例如 Inference Endpoints 或 SageMaker 时更合适。它与 `transformers`、HF Hub 集成紧密，如果团队本身就是 HF 技术栈，通常是进入生产环境最直接的路径。

- **SGLang**：当智能体有大量多轮推理，例如 tree-of-thought、自一致性（self-consistency）、分支式工具调用时尤其值得考虑。RadixAttention 能比 vLLM 的线性前缀缓存更高效地复用不同对话分支之间的 KV cache；它的压缩有限状态机方案也非常适合结构化输出。

In [ ]:
%pip install -q vllm openai

In [ ]:
import json, os, time, subprocess, signal, statistics
from pathlib import Path
from openai import OpenAI
import requests

# ── 模型配置（Model configuration）───────────────────────────────────
# 可以替换为任何当前 GPU 显存能够容纳的模型。
# Qwen2.5-3B 可放入单张 T4（16 GB）；若使用 A100，可以尝试 8B 或 14B 模型。
VLLM_MODEL       = "Qwen/Qwen2.5-3B-Instruct"
SERVED_NAME       = "qwen"

# ── 引擎参数（Engine args）────────────────────────────────────────────
# 这些参数与 vLLM 的 engine arguments 直接对应：
#   https://docs.vllm.ai/en/stable/configuration/engine_args/
VLLM_PORT              = 8001
VLLM_HOST              = "0.0.0.0"
TENSOR_PARALLEL_SIZE   = 1        # number of GPUs for tensor parallelism
GPU_MEMORY_UTILIZATION = 0.90     # fraction of GPU VRAM vLLM may use
MAX_MODEL_LEN          = 4096     # max context window (tokens)
DTYPE                  = "auto"   # "auto" | "float16" | "bfloat16"
SWAP_SPACE             = 4        # GiB of CPU swap for KV cache overflow
MAX_NUM_SEQS           = 64       # max concurrent sequences in a batch
DISABLE_LOG_STATS      = True     # quieter logs in notebook

# OpenAI-compatible 客户端——与 OpenRouter、Azure 或其他托管 API 使用相同接口。
# 这正是统一接口的价值：切换推理后端时，无需修改智能体代码。
local_client = OpenAI(
    base_url=f"http://localhost:{VLLM_PORT}/v1",
    api_key="not-needed",  # 本地服务器，无需鉴权
)

print(f"Model:  {VLLM_MODEL}")
print(f"Server: http://localhost:{VLLM_PORT}/v1")
print(f"Engine: TP={TENSOR_PARALLEL_SIZE}, GPU mem={GPU_MEMORY_UTILIZATION}, "
      f"max_len={MAX_MODEL_LEN}, dtype={DTYPE}")

Model:  Qwen/Qwen2.5-3B-Instruct
Server: http://localhost:8001/v1
Engine: TP=1, GPU mem=0.9, max_len=4096, dtype=auto


## 1 — 冷启动（Cold Start）：从 `vllm serve` 到第一个 Token

冷启动并不只是“服务器启动需要一段时间”，而是一连串会阻塞服务的步骤；在这些步骤完成之前，智能体无法为第一个用户提供正常响应：

```
 ┌────────────────┐   ┌──────────────────┐   ┌──────────────────────┐   ┌─────────────┐
 │ 下载模型权重   │ → │ 加载进 VRAM      │ → │ 首次请求：分配      │ → │ 预热后请求  │
 │（若本地未缓存）│   │（反序列化并在    │   │ KV cache、编译剩余  │   │（稳态）     │
 │                │   │ GPU 间分片）      │   │ kernel、预热调度器   │   │             │
 │ ~0-60s         │   │ ~15-60s          │   │ ~1-5s 额外延迟      │   │ ~0.2-1s     │
 └────────────────┘   └──────────────────┘   └──────────────────────┘   └─────────────┘
      阶段 1                 阶段 2                  阶段 3                阶段 4
               ──────── cold_start_seconds ────────  ─ 首次请求 ─  ── 预热态 ──
```

这里会做端到端测量：从零启动服务器，等待健康检查通过，再依次测量前几次请求，从而看到从冷启动到稳态的完整过渡。

我们通过 `python -m vllm.entrypoints.openai.api_server` 启动 vLLM——这是暴露 `/v1/chat/completions` 的 **OpenAI-compatible（OpenAI 兼容）**入口，底层与 `vllm serve` 一致。各参数直接对应 vLLM 的 **[Engine Arguments](https://docs.vllm.ai/en/stable/configuration/engine_args/)**：

| 参数 | 控制什么 | 为什么重要 |
|---|---|---|
| `--model` | HF 模型 ID 或本地路径 | 决定加载到 GPU VRAM 的模型 |
| `--tensor-parallel-size` | Tensor Parallel 使用的 GPU 数量 | 决定如何跨 GPU 切分模型 |
| `--gpu-memory-utilization` | 分配给 KV cache 的 VRAM 比例 | 太高可能 OOM；太低会减少可并发序列数 |
| `--max-model-len` | 最大上下文窗口 | 限制单次请求可占用的 KV cache |
| `--swap-space` | 用于 KV 溢出的 CPU RAM（GiB） | GPU cache 满时提供安全缓冲 |
| `--dtype` | 计算精度 | `auto` 会根据 GPU 选择 bf16 / fp16 |
| `--max-num-seqs` | 最大并发序列数 | 限制显存占用并控制 batching |
| `--enforce-eager` | 禁用 CUDA graph capture | 启动更快，但吞吐略低 |
| `--disable-log-stats` | 关闭周期性统计日志 | 让 Notebook 日志更干净 |

In [ ]:
import subprocess, time, requests

VLLM_LOG = "vllm_server.log"


def build_vllm_args(
    model: str = VLLM_MODEL,
    served_name: str = SERVED_NAME,
    port: int = VLLM_PORT,
    extra_args: list[str] | None = None,
) -> list[str]:
    """Build the vLLM server command using python -m vllm.entrypoints.openai.api_server.

    Every flag here maps to a vLLM Engine Argument:
    https://docs.vllm.ai/en/stable/configuration/engine_args/
    """
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        f"--host={VLLM_HOST}",
        f"--port={port}",
        f"--model={model}",
        f"--served-model-name={served_name}",
        f"--tensor-parallel-size={TENSOR_PARALLEL_SIZE}",
        f"--gpu-memory-utilization={GPU_MEMORY_UTILIZATION}",
        f"--max-model-len={MAX_MODEL_LEN}",
        f"--dtype={DTYPE}",
        f"--swap-space={SWAP_SPACE}",
        f"--max-num-seqs={MAX_NUM_SEQS}",
        "--enforce-eager",          # faster startup, skip CUDA graph capture
    ]
    if DISABLE_LOG_STATS:
        cmd.append("--disable-log-stats")
    if extra_args:
        cmd.extend(extra_args)
    return cmd


def start_vllm_server(
    model: str = VLLM_MODEL,
    served_name: str = SERVED_NAME,
    port: int = VLLM_PORT,
    extra_args: list[str] | None = None,
) -> subprocess.Popen:
    """Start a vLLM server process and return the handle."""
    cmd = build_vllm_args(model, served_name, port, extra_args)

    log_fh = open(VLLM_LOG, "w")
    proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

    # Pretty-print the launch command
    print(f"  vLLM PID {proc.pid}")
    print(f"  cmd: \\\n    " + " \\\n    ".join(cmd))
    return proc


def wait_for_healthy(port: int = VLLM_PORT, timeout: int = 300) -> float:
    """Block until the vLLM health endpoint responds. Returns seconds waited."""
    url = f"http://localhost:{port}/health"
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            r = requests.get(url, timeout=2)
            if r.status_code == 200:
                elapsed = time.time() - t0
                print(f"  ✅ Server healthy after {elapsed:.1f}s")
                return elapsed
        except requests.ConnectionError:
            pass
        time.sleep(2)
    raise TimeoutError(f"vLLM not healthy after {timeout}s — check {VLLM_LOG}")


def stop_vllm_server(proc: subprocess.Popen):
    """Gracefully stop the vLLM server."""
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    print(f"  Server stopped (PID {proc.pid})")

In [ ]:
SYSTEM_PROMPT = (
    "You are a customer support triage agent for an e-commerce company. "
    "Classify tickets by category and priority. Be concise."
)

TEST_MESSAGE = "I placed order #ORD-8842 three weeks ago and nobody has told me anything. This is unacceptable."


def timed_completion(
    client: OpenAI,
    messages: list[dict],
    model: str = SERVED_NAME,
    max_tokens: int = 256,
) -> dict:
    """Send a chat completion and return timing + usage metadata."""
    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.2,
    )
    elapsed = time.time() - t0

    usage = response.usage
    content = response.choices[0].message.content or ""

    return {
        "latency_s": round(elapsed, 3),
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "total_tokens": usage.total_tokens if usage else 0,
        "content": content,
        "tokens_per_sec": round(
            (usage.completion_tokens / elapsed) if usage and elapsed > 0 else 0, 1
        ),
    }



#  PHASE 1+2: Start server from scratch, measure boot time

print("=" * 70)
print(" COLD START — FULL END-TO-END MEASUREMENT")
print("=" * 70)

t_total_start = time.time()

print("\n  Phase 1+2: Starting vLLM server (model download + GPU loading)...\n")
vllm_proc = start_vllm_server()
cold_start_seconds = wait_for_healthy()

print(f"\n  ⏱  Server boot (Phase 1+2): {cold_start_seconds:.1f}s")
print(f"     Model loaded into VRAM, health endpoint responding.\n")


#  PHASE 3+4: First requests — cold GPU → warm steady state

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": TEST_MESSAGE},
]

results = []
labels = [
    "1st request (cold GPU)  ← Phase 3",
    "2nd request (warming)",
    "3rd request (warm)      ← Phase 4",
    "4th request (warm)",
    "5th request (warm)",
]

print("  Phase 3+4: Request latencies after boot:")
print("  " + "─" * 66)

for i, label in enumerate(labels):
    r = timed_completion(local_client, messages)
    results.append(r)
    print(
        f"  {label:<35} │ {r['latency_s']:>6.3f}s │ "
        f"{r['completion_tokens']:>4} tok │ "
        f"{r['tokens_per_sec']:>6.1f} tok/s"
    )

t_total_first_response = time.time() - t_total_start

# Summary
cold_req      = results[0]["latency_s"]
warm_avg      = statistics.mean(r["latency_s"] for r in results[2:])  # 3rd-5th
total_to_first = cold_start_seconds + cold_req

print(f"\n{'═' * 70}")
print(f"  COLD START BREAKDOWN")
print(f"{'─' * 70}")
print(f"  Server boot (Phase 1+2):   {cold_start_seconds:>7.1f}s   model → VRAM")
print(f"  1st request  (Phase 3):    {cold_req:>7.3f}s   KV alloc + kernel compile")
print(f"  ────────────────────────── ─────────")
print(f"  Total cold start:          {total_to_first:>7.1f}s   deploy → first answer")
print(f"")
print(f"  Warm request (Phase 4):    {warm_avg:>7.3f}s   steady state")
print(f"  Cold/Warm ratio:           {total_to_first / warm_avg:>7.0f}×   ← your user waits this much longer")
print(f"{'─' * 70}")
print(f"  → After every deploy, restart, or scale-from-zero event, your agent")
print(f"    is OFFLINE for ~{total_to_first:.0f}s. Plan for it: health checks, warm-up")
print(f"    probes, min-replicas > 0, or rolling deploys.")

 COLD START — FULL END-TO-END MEASUREMENT

  Phase 1+2: Starting vLLM server (model download + GPU loading)...

  vLLM PID 3941
  cmd: \
    python \
    -m \
    vllm.entrypoints.openai.api_server \
    --host=0.0.0.0 \
    --port=8001 \
    --model=Qwen/Qwen2.5-3B-Instruct \
    --served-model-name=qwen \
    --tensor-parallel-size=1 \
    --gpu-memory-utilization=0.9 \
    --max-model-len=4096 \
    --dtype=auto \
    --swap-space=4 \
    --max-num-seqs=64 \
    --enforce-eager \
    --disable-log-stats
  ✅ Server healthy after 124.1s

  ⏱  Server boot (Phase 1+2): 124.1s
     Model loaded into VRAM, health endpoint responding.

  Phase 3+4: Request latencies after boot:
  ──────────────────────────────────────────────────────────────────
  1st request (cold GPU)  ← Phase 3   │  0.458s │   10 tok │   21.9 tok/s
  2nd request (warming)               │  0.276s │   10 tok │   36.2 tok/s
  3rd request (warm)      ← Phase 4   │  0.277s │   10 tok │   36.1 tok/s
  4th request (warm)      

# 上下文长度与 KV Cache 显存压力（Context Length and KV Cache Pressure）

> **说明：** 本节专门讨论 **decoder-only（仅解码器、自回归）**模型，例如 GPT、LLaMA、Qwen、Mistral 等——这也是 vLLM、TGI 和 SGLang 主要服务的模型类型。Encoder-decoder（编码器-解码器）架构，例如 T5、BART，会先计算一次 encoder KV cache，并在后续所有解码步骤中复用，因此 Prompt 长度对其单 token 生成延迟的影响要小得多。

对于 decoder-only 模型，Prompt 中的每个 token 都会生成一组 KV pair（键值对），这些数据既需要保存在 GPU 显存里，又会在后续每一步生成时被 attention 访问。随着智能体的对话历史越来越长，会出现三件事：

1. **TTFT（Time to First Token，首 token 延迟）上升** —— prefill（预填充）阶段必须对完整 Prompt 计算 attention，其计算成本会随序列长度近似二次增长。
2. **显存压力上升** —— KV cache 会随上下文长度线性增长，从而压缩同时批处理其他请求的空间。
3. **单 token 解码变慢** —— 每生成一个新 token，都需要访问此前全部 KV pair；上下文越长，每一步需要读取的显存数据就越多。

因此，前一个 Notebook 讨论的 context pruning（上下文裁剪）并不只是为了省钱，它同样是在维持推理后端的响应能力。

In [ ]:
# ── 构造上下文长度逐步增加的 Prompt ──────────────────────────────────
FILLER_TURN = (
    "The customer previously wrote: 'I ordered a blue widget on January 5th "
    "and the tracking says it's stuck in Memphis. Can someone look into this? "
    "I also have a question about your return policy for items over $50.'"
)

def build_messages(num_history_turns: int) -> list[dict]:
    """构造包含 N 轮填充对话，再追加真实问题的会话。"""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for i in range(num_history_turns):
        msgs.append({"role": "user", "content": f"[Turn {i+1}] {FILLER_TURN}"})
        msgs.append({"role": "assistant", "content": f"Acknowledged turn {i+1}."})
    msgs.append({"role": "user", "content": TEST_MESSAGE})
    return msgs


# ── 测量不同上下文长度下的延迟 ───────────────────────────────────────
turn_counts = [0, 5, 10, 20, 30]
context_results = []

print("=" * 70)
print(" CONTEXT LENGTH vs. LATENCY")
print("=" * 70)

for n in turn_counts:
    msgs = build_messages(n)
    r = timed_completion(local_client, msgs, max_tokens=128)
    context_results.append({"turns": n, **r})
    print(
        f"  {n:>3} history turns │ {r['prompt_tokens']:>5} prompt tok │ "
        f"{r['latency_s']:>6.3f}s │ {r['tokens_per_sec']:>6.1f} tok/s"
    )

# ── 计算延迟放大倍数 ──────────────────────────────────────────────────
baseline = context_results[0]["latency_s"]
heaviest = context_results[-1]["latency_s"]
print(f"\n{'─' * 70}")
print(f"  Baseline (0 turns):   {baseline:.3f}s")
print(f"  Heaviest ({turn_counts[-1]} turns): {heaviest:.3f}s")
print(f"  Slowdown:             {heaviest / baseline:.1f}×")
print(f"\n  → Long agent conversations get slower even if the final question is identical.")

 CONTEXT LENGTH vs. LATENCY
    0 history turns │    62 prompt tok │  0.282s │   35.4 tok/s
    5 history turns │   407 prompt tok │  0.223s │   31.4 tok/s
   10 history turns │   754 prompt tok │  0.286s │   34.9 tok/s
   20 history turns │  1464 prompt tok │  0.232s │   30.2 tok/s
   30 history turns │  2174 prompt tok │  0.309s │   32.4 tok/s

──────────────────────────────────────────────────────────────────────
  Baseline (0 turns):   0.282s
  Heaviest (30 turns): 0.309s
  Slowdown:             1.1×

  → Long agent conversations get slower even if the final question is identical.


# 前缀缓存（Prefix Caching）：A/B 对比

大多数智能体在每次请求中都会使用**相同的 system prompt**。如果没有 prefix caching，服务器会在每一次调用时重新计算这段 system prompt 的 KV cache——这完全是重复开销。

vLLM 的 `--enable-prefix-caching`（APC，Automatic Prefix Caching，自动前缀缓存）会检测多个请求是否共享相同前缀，并复用已经缓存的 KV blocks：

```
请求 1： [SYSTEM: "You are a support agent..."] + [USER: "My order is late"]
                    ↑ 计算 KV，并写入缓存

请求 2： [SYSTEM: "You are a support agent..."] + [USER: "Refund policy?"]
                    ↑ cache HIT（缓存命中）→ 跳过这一前缀的 KV 计算
```

为了观察真实差异，我们会把**完全相同的 5 个问题**各跑一遍：
1. **关闭** prefix caching（当前服务器）
2. **开启** `--enable-prefix-caching`（重启服务器）

然后把两组结果并排比较。

In [ ]:
#  5 test questions (same system prompt, different user messages) ─
user_messages = [
    "I placed order #ORD-8842 three weeks ago and nobody has told me anything.",
    "What is your return policy for electronics?",
    "I need to cancel order #ORD-1234 immediately.",
    "My package arrived damaged — the box was crushed.",
    "Can I change the shipping address on my pending order?",
]


def run_prefix_experiment(client, label: str) -> list[dict]:
    """Send the same 5 questions and collect latency results."""
    # Warm-up request (not counted) — primes GPU scheduler / KV allocator
    _ = timed_completion(client, messages, max_tokens=16)
    time.sleep(0.5)

    results = []
    for i, user_msg in enumerate(user_messages):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        r = timed_completion(client, msgs, max_tokens=128)
        results.append(r)
        print(
            f"  Q{i+1}: {r['latency_s']:>6.3f}s │ "
            f"{r['prompt_tokens']:>4} prompt tok │ "
            f"{r['tokens_per_sec']:>6.1f} tok/s │ "
            f"{user_msg[:50]}..."
        )
    return results



#  RUN A: WITHOUT prefix caching (current server)

print("=" * 70)
print(" RUN A — WITHOUT prefix caching")
print("=" * 70)
results_no_cache = run_prefix_experiment(local_client, "no-cache")


#  Restart server WITH prefix caching
print(f"\n{'─' * 70}")
print("Stopping current server...")
stop_vllm_server(vllm_proc)
time.sleep(3)

print("\nStarting vLLM server WITH --enable-prefix-caching...")
vllm_proc = start_vllm_server(extra_args=["--enable-prefix-caching"])
wait_for_healthy()


#  RUN B: WITH prefix caching

print("\n" + "=" * 70)
print(" RUN B — WITH prefix caching")
print("=" * 70)
results_with_cache = run_prefix_experiment(local_client, "with-cache")


#  SIDE-BY-SIDE COMPARISON

print("\n" + "=" * 70)
print(" A/B COMPARISON — prefix caching OFF vs ON")
print("=" * 70)
print(f"  {'Q':<3} │ {'Without':>9} │ {'With':>9} │ {'Speedup':>8} │ Question")
print(f"  {'─'*3}─┼─{'─'*9}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*40}")

for i, (no_c, with_c) in enumerate(zip(results_no_cache, results_with_cache)):
    speedup = no_c["latency_s"] / with_c["latency_s"] if with_c["latency_s"] > 0 else 0
    print(
        f"  Q{i+1}  │ {no_c['latency_s']:>8.3f}s │ {with_c['latency_s']:>8.3f}s │ "
        f"{speedup:>7.2f}× │ {user_messages[i][:40]}..."
    )

avg_no   = statistics.mean(r["latency_s"] for r in results_no_cache)
avg_with = statistics.mean(r["latency_s"] for r in results_with_cache)
avg_speedup = avg_no / avg_with if avg_with > 0 else 0

# Use results from Q2-Q5 for "cached" comparison (Q1 primes the prefix cache)
avg_no_2_5   = statistics.mean(r["latency_s"] for r in results_no_cache[1:])
avg_with_2_5 = statistics.mean(r["latency_s"] for r in results_with_cache[1:])
cached_speedup = avg_no_2_5 / avg_with_2_5 if avg_with_2_5 > 0 else 0

print(f"\n{'─' * 70}")
print(f"  Avg (all 5):   {avg_no:.3f}s → {avg_with:.3f}s  ({avg_speedup:.2f}× speedup)")
print(f"  Avg (Q2-Q5):   {avg_no_2_5:.3f}s → {avg_with_2_5:.3f}s  ({cached_speedup:.2f}× speedup)")
print(f"\n  → Q1 primes the prefix cache. From Q2 onward the system prompt KV is")
print(f"    reused — that's the steady-state benefit your agent gets in production.")

 RUN A — WITHOUT prefix caching
  Q1:  0.304s │   58 prompt tok │   32.9 tok/s │ I placed order #ORD-8842 three weeks ago and nobod...
  Q2:  1.033s │   46 prompt tok │   35.8 tok/s │ What is your return policy for electronics?...
  Q3:  0.228s │   52 prompt tok │   30.6 tok/s │ I need to cancel order #ORD-1234 immediately....
  Q4:  0.282s │   48 prompt tok │   35.4 tok/s │ My package arrived damaged — the box was crushed....
  Q5:  0.258s │   49 prompt tok │   34.9 tok/s │ Can I change the shipping address on my pending or...

──────────────────────────────────────────────────────────────────────
Stopping current server...
  Server stopped (PID 3941)

Starting vLLM server WITH --enable-prefix-caching...
  vLLM PID 4853
  cmd: \
    python \
    -m \
    vllm.entrypoints.openai.api_server \
    --host=0.0.0.0 \
    --port=8001 \
    --model=Qwen/Qwen2.5-3B-Instruct \
    --served-model-name=qwen \
    --tensor-parallel-size=1 \
    --gpu-memory-utilization=0.9 \
    --max-model-len=40

# 缓存失效（Cache Invalidation）：当前缀发生变化

前缀缓存非常强大——前提是前缀没有变化。在智能体系统中，下面这些情况都会改变前缀：

- **修改 system prompt**（A/B 测试、角色切换）
- **注入动态上下文**（RAG 检索、用户画像、工具结果）
- **让对话产生分叉**（智能体分支路径）

每一次前缀变化都会造成一次 **cache miss（缓存未命中）**——服务器必须为新的前缀从头重新计算 KV cache。这就是所谓的“缓存失效税（cache invalidation tax）”。

In [ ]:
# ── Different system prompts = cache misses ───────────────────────────
SYSTEM_PROMPTS = {
    "support_agent": (
        "You are a customer support triage agent for an e-commerce company. "
        "Classify tickets by category and priority. Be concise."
    ),
    "returns_specialist": (
        "You are a returns and refund specialist. Help customers with return "
        "eligibility, refund timelines, and exchange options. Be precise about "
        "policy details and timelines."
    ),
    "escalation_agent": (
        "You are an escalation manager who handles VIP customers and complex "
        "cases. Use a formal, empathetic tone. Offer concrete next steps "
        "and set clear expectations for resolution timelines."
    ),
    "back_to_support": (
        "You are a customer support triage agent for an e-commerce company. "
        "Classify tickets by category and priority. Be concise."
    ),
}

fixed_user_msg = "My order hasn't arrived and I'm really frustrated."

print("=" * 70)
print(" CACHE INVALIDATION — CHANGING THE SYSTEM PROMPT")
print("=" * 70)

invalidation_results = {}
for label, sys_prompt in SYSTEM_PROMPTS.items():
    msgs = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": fixed_user_msg},
    ]
    r = timed_completion(local_client, msgs, max_tokens=128)
    invalidation_results[label] = r
    cache_status = "MISS" if label != "back_to_support" else "HIT (same as #1)"
    print(
        f"  {label:<22} │ {r['latency_s']:>6.3f}s │ "
        f"{r['tokens_per_sec']:>6.1f} tok/s │ cache: {cache_status}"
    )

# ── Highlight the cost of switching ──────────────────────────────────
first = invalidation_results["support_agent"]["latency_s"]
back  = invalidation_results["back_to_support"]["latency_s"]
print(f"\n{'─' * 70}")
print(f"  First 'support_agent':    {first:.3f}s  (cache miss → compute KV)")
print(f"  Return to same prompt:    {back:.3f}s   (cache hit → reuse KV)")
print(f"\n  → Dynamic system prompts (RAG injection, role switching) invalidate")
print(f"    the prefix cache. Design for stable prefixes where possible.")

 CACHE INVALIDATION — CHANGING THE SYSTEM PROMPT
  support_agent          │  0.283s │   35.3 tok/s │ cache: MISS
  returns_specialist     │  3.253s │   39.4 tok/s │ cache: MISS
  escalation_agent       │  3.267s │   39.2 tok/s │ cache: MISS
  back_to_support        │  0.315s │   34.9 tok/s │ cache: HIT (same as #1)

──────────────────────────────────────────────────────────────────────
  First 'support_agent':    0.283s  (cache miss → compute KV)
  Return to same prompt:    0.315s   (cache hit → reuse KV)

  → Dynamic system prompts (RAG injection, role switching) invalidate
    the prefix cache. Design for stable prefixes where possible.


# 对话增长（Conversation Growth）：多轮交互中的缓存行为

在真实的智能体循环中，对话会一轮一轮增长。启用 prefix caching（前缀缓存）后，每一轮新增请求只需要为**新加入的 token**计算 KV；此前的整个前缀（system prompt + 所有历史轮次）都已经缓存。

但当对话发生*分叉*时（例如智能体改用另一种工具调用进行重试），分叉点之后的后缀缓存就会失效。这正体现了“线性对话（linear conversation，缓存友好）”与“思维树（tree-of-thought，缓存不友好）”之间的权衡。

In [ ]:
# ── Simulate a growing conversation (linear) ─────────────────────────
conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

turn_pairs = [
    ("My order #ORD-8842 hasn't arrived.", "I'll look into that for you."),
    ("It's been three weeks now.", "I understand your frustration. Let me check the status."),
    ("Can you expedite the shipping?", "I'll escalate this to our logistics team."),
    ("What about a refund if it doesn't arrive by Friday?", "Our policy allows refunds after 21 business days."),
    ("Okay, can you also check order #ORD-9901?", "Sure, let me pull up that order as well."),
    ("That one has the wrong item. I got a red widget instead of blue.", "I'm sorry about that. Let me initiate a return for the incorrect item."),
]

print("=" * 70)
print(" MULTI-TURN CONVERSATION — LINEAR GROWTH (cache-friendly)")
print("=" * 70)

linear_results = []
for i, (user_msg, assistant_msg) in enumerate(turn_pairs):
    conversation.append({"role": "user", "content": user_msg})
    r = timed_completion(local_client, conversation, max_tokens=128)
    linear_results.append({"turn": i + 1, **r})
    # Add the assistant response to keep the conversation going
    conversation.append({"role": "assistant", "content": assistant_msg})
    print(
        f"  Turn {i+1} │ {r['prompt_tokens']:>5} prompt tok │ "
        f"{r['latency_s']:>6.3f}s │ {r['tokens_per_sec']:>6.1f} tok/s"
    )

# ── Now fork: same base conversation, different last message ─────────
print(f"\n{'─' * 70}")
print(" CONVERSATION FORK — cache invalidation at the branch point")
print("─" * 70)

# Fork A: continue the original conversation
fork_a = conversation.copy()
fork_a.append({"role": "user", "content": "Actually, forget the return. Just send me a replacement."})

# Fork B: different branch from the same point
fork_b = conversation.copy()
fork_b.append({"role": "user", "content": "I want to speak with a supervisor about both orders."})

r_a = timed_completion(local_client, fork_a, max_tokens=128)
r_b = timed_completion(local_client, fork_b, max_tokens=128)

print(f"  Fork A (replacement): {r_a['latency_s']:>6.3f}s │ {r_a['prompt_tokens']} prompt tok")
print(f"  Fork B (supervisor):  {r_b['latency_s']:>6.3f}s │ {r_b['prompt_tokens']} prompt tok")
print(f"\n  → Both forks share the same prefix and differ only in the last message.")
print(f"    With prefix caching, the shared history is not recomputed.")

 MULTI-TURN CONVERSATION — LINEAR GROWTH (cache-friendly)
  Turn 1 │    51 prompt tok │  0.281s │   35.6 tok/s
  Turn 2 │    76 prompt tok │  0.279s │   35.8 tok/s
  Turn 3 │   104 prompt tok │  0.525s │   38.1 tok/s
  Turn 4 │   135 prompt tok │  0.661s │   37.8 tok/s
  Turn 5 │   171 prompt tok │  0.502s │   37.9 tok/s
  Turn 6 │   208 prompt tok │  0.272s │   33.1 tok/s

──────────────────────────────────────────────────────────────────────
 CONVERSATION FORK — cache invalidation at the branch point
──────────────────────────────────────────────────────────────────────
  Fork A (replacement):  0.343s │ 246 prompt tok
  Fork B (supervisor):   0.336s │ 245 prompt tok

  → Both forks share the same prefix and differ only in the last message.
    With prefix caching, the shared history is not recomputed.


# 后端可移植性（Backend Portability）：同一个客户端，不同的服务器

一个关键的部署原则是：**智能体代码应该与具体后端解耦（backend-agnostic）。** 由于三种后端都暴露 OpenAI-compatible API（OpenAI 兼容接口），因此切换后端应该只是配置变化，而不是代码变化。下面会展示同一个 `OpenAI` 客户端构造方式如何连接不同后端。

In [ ]:
# ── Backend configurations — same OpenAI client, different base_url ───
#
# Each backend uses its own entrypoint / CLI, but they all expose
# an OpenAI-compatible /v1/chat/completions endpoint.
#
# vLLM engine args reference:
#   https://docs.vllm.ai/en/stable/configuration/engine_args/

MODEL_EXAMPLE = "Qwen/Qwen2.5-3B-Instruct"

BACKEND_CONFIGS = {
    "vLLM": {
        "base_url": "http://localhost:8001/v1",
        "start_cmd": [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            f"--host=0.0.0.0",
            f"--port=8001",
            f"--model={MODEL_EXAMPLE}",
            f"--served-model-name=qwen",
            f"--tensor-parallel-size=1",
            f"--gpu-memory-utilization=0.90",
            f"--max-model-len=4096",
            f"--dtype=auto",
            f"--swap-space=4",
            f"--max-num-seqs=64",
            "--enforce-eager",
            "--enable-prefix-caching",
            "--disable-log-stats",
        ],
        "docs": "https://docs.vllm.ai/en/stable/configuration/engine_args/",
    },
    "TGI": {
        "base_url": "http://localhost:8002/v1",
        "start_cmd": [
            "text-generation-launcher",
            f"--model-id={MODEL_EXAMPLE}",
            "--port=8002",
            "--hostname=0.0.0.0",
            "--max-input-tokens=4096",
            "--max-total-tokens=4608",
            "--dtype=float16",
        ],
        "docs": "https://huggingface.co/docs/text-generation-inference",
    },
    "SGLang": {
        "base_url": "http://localhost:8003/v1",
        "start_cmd": [
            "python", "-m", "sglang.launch_server",
            f"--model-path={MODEL_EXAMPLE}",
            f"--served-model-name=qwen",
            "--port=8003",
            "--host=0.0.0.0",
        ],
        "docs": "https://docs.sglang.ai/",
    },
}

print("=" * 70)
print(" BACKEND PORTABILITY — YOUR AGENT CODE DOESN'T CHANGE")
print("=" * 70)

for name, cfg in BACKEND_CONFIGS.items():
    cmd_str = " \\\n              ".join(cfg["start_cmd"])
    print(f"\n  ── {name} ──")
    print(f"  Docs:     {cfg['docs']}")
    print(f"  Start:    {cmd_str}")
    print(f"  Client:   OpenAI(base_url='{cfg['base_url']}', api_key='not-needed')")
    print(f"  Call:     client.chat.completions.create(model='qwen', messages=...)")

print(f"\n{'─' * 70}")
print("  → Same chat.completions.create() call. Swap the base_url and you've")
print("    switched your entire inference backend. No agent code changes.")
print("    This is why using the OpenAI SDK as your client library matters —")
print("    it's the lingua franca of inference APIs.")

 BACKEND PORTABILITY — YOUR AGENT CODE DOESN'T CHANGE

  ── vLLM ──
  Docs:     https://docs.vllm.ai/en/stable/configuration/engine_args/
  Start:    python \
              -m \
              vllm.entrypoints.openai.api_server \
              --host=0.0.0.0 \
              --port=8001 \
              --model=Qwen/Qwen2.5-3B-Instruct \
              --served-model-name=qwen \
              --tensor-parallel-size=1 \
              --gpu-memory-utilization=0.90 \
              --max-model-len=4096 \
              --dtype=auto \
              --swap-space=4 \
              --max-num-seqs=64 \
              --enforce-eager \
              --enable-prefix-caching \
              --disable-log-stats
  Client:   OpenAI(base_url='http://localhost:8001/v1', api_key='not-needed')
  Call:     client.chat.completions.create(model='qwen', messages=...)

  ── TGI ──
  Docs:     https://huggingface.co/docs/text-generation-inference
  Start:    text-generation-launcher \
              --model-id=Qwen/Q

## 汇总看板（Summary Dashboard）

In [ ]:
print("=" * 70)
print(" DEPLOYMENT PHYSICS — SUMMARY")
print("=" * 70)

print(f"""
  ┌─────────────────────────────────────────────────────────────────┐
  │  COLD START                                                     │
  │  Server boot:            {cold_start_seconds:>6.1f}s                              │
  │  1st request:            {cold_req:>6.3f}s                              │
  │  Total (boot + 1st req): {total_to_first:>6.1f}s                              │
  │  Warm request avg:       {warm_avg:>6.3f}s                              │
  │  Cold/Warm ratio:        {total_to_first / warm_avg:>5.0f}×                               │
  │                                                                 │
  │  CONTEXT LENGTH                                                 │
  │  0 turns  → {context_results[0]['latency_s']:>6.3f}s                                      │
  │  {turn_counts[-1]} turns → {context_results[-1]['latency_s']:>6.3f}s  ({context_results[-1]['latency_s'] / context_results[0]['latency_s']:.1f}× slower)                        │
  │                                                                 │
  │  PREFIX CACHING (A/B)                                           │
  │  Without (avg Q2-5): {avg_no_2_5:>6.3f}s                                 │
  │  With    (avg Q2-5): {avg_with_2_5:>6.3f}s  ({cached_speedup:.2f}× faster)              │
  │                                                                 │
  │  CACHE INVALIDATION                                             │
  │  Same prefix (return):   {invalidation_results['back_to_support']['latency_s']:>6.3f}s (cache hit)               │
  │  New prefix (switch):    {invalidation_results['returns_specialist']['latency_s']:>6.3f}s (cache miss)              │
  │                                                                 │
  │  MODEL:  {VLLM_MODEL:<40}            │
  └─────────────────────────────────────────────────────────────────┘
""")

 DEPLOYMENT PHYSICS — SUMMARY

  ┌─────────────────────────────────────────────────────────────────┐
  │  COLD START                                                     │
  │  Server boot:             124.1s                              │
  │  1st request:             0.458s                              │
  │  Total (boot + 1st req):  124.6s                              │
  │  Warm request avg:        0.277s                              │
  │  Cold/Warm ratio:          449×                               │
  │                                                                 │
  │  CONTEXT LENGTH                                                 │
  │  0 turns  →  0.282s                                      │
  │  30 turns →  0.309s  (1.1× slower)                        │
  │                                                                 │
  │  PREFIX CACHING (A/B)                                           │
  │  Without (avg Q2-5):  0.450s                                 │
  │  With    (a

# 清理（Cleanup）

In [ ]:
# ── Stop the vLLM server ──────────────────────────────────────────────
stop_vllm_server(vllm_proc)

# ── Clean up log file ─────────────────────────────────────────────────
if os.path.exists(VLLM_LOG):
    os.remove(VLLM_LOG)
    print(f"  Removed {VLLM_LOG}")

print("  Done.")

  Server stopped (PID 4853)
  Removed vllm_server.log
  Done.
